In [0]:
import pandas as pd
import requests
import json
import os
from datetime import datetime
from dotenv import load_dotenv

# --- CONFIGURAÇÃO ---
load_dotenv()

WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")

def limpar_texto_ia(texto):
    """Remove repetições e limpa o texto gerado pela IA."""
    if not texto: return "Diagnóstico indisponível."
    # Remove prefixos comuns de chat
    texto = texto.replace("Diagnóstico IA:", "").strip()
    # Garante que termine com pontuação
    if texto[-1] not in ".!?": texto += "."
    return texto

def enviar_alerta_discord(webhook_url, categoria, taxa, diagnostico):
    """Envia um card bonito pro Discord (Opcional)"""
    data = {
        "embeds": [{
            "title": f"BLOQUEIO PREVENTIVO: {categoria}",
            "color": 15548997,
            "fields": [
                {"name": "Taxa de Reprovação", "value": f"**{taxa:.1f}%** (Crítico > 40%)", "inline": True},
                {"name": "Diagnóstico da IA", "value": diagnostico[:1000]} # Discord tem limite
            ],
            "footer": {"text": "Olist CX Intelligence • Action Engine"},
            "timestamp": datetime.utcnow().isoformat()
        }]
    }
    try:
        requests.post(webhook_url, json=data)
    except Exception as e:
        print(f"Erro no envio: {e}")

# --- 1. LEITURA DA INTELEGIÊNCIA (GOLD) ---
print("Conectando ao Cérebro de IA (Camada Gold)...")
df_analise = spark.read.table("olist_portfolio.gold.ai_diagnostics").toPandas()

# Aplica limpeza
df_analise['diagnostico_ia'] = df_analise['diagnostico_ia'].astype(str).apply(limpar_texto_ia)

print(f"{len(df_analise)} Alertas Críticos Encontrados.")

# --- 2. MOTOR DE DECISÃO ---
print("\n" + "═"*80)
print(f"EXECUTANDO PROTOCOLO DE CONTENÇÃO")
print("═"*80)

for index, row in df_analise.iterrows():
    cat = row['categoria'].upper()
    taxa = row['taxa_reprovacao']*100
    diag = row['diagnostico_ia']
    
    # Simulação do E-mail (O que vai pro Log)
    print(f"\nALVO: {cat}")
    print(f"   Taxa Atual: {taxa:.1f}%")
    
    email_simulado = f"""
    PARA: gestao.comercial@olist.com
    ASSUNTO: SUSPENSÃO IMEDIATA: Categoria {cat}
    
    Prezados,
    
    O motor de risco identificou uma anomalia crítica.
    
    MOTIVO DO BLOQUEIO (ANÁLISE IA):
    "{diag}"
    
    AÇÃO AUTOMÁTICA:
    - Vendas suspensas para novos pedidos.
    - Fornecedores notificados.
    """
    
    print("-" * 40)
    print(email_simulado)
    print("-" * 40)
    
    # Disparo Real (Se tiver Webhook)
    if WEBHOOK_URL:
        print("   Enviando sinal para HQ (Discord)...")
        enviar_alerta_discord(WEBHOOK_URL, cat, taxa, diag)
        print("   Sinal Enviado.")
    else:
        print("   Modo Simulação: Nenhum sinal externo enviado.")

print("\nProcesso de Auditoria Finalizado.")